In [35]:
# 1: 설치
!pip install ultralytics
!pip install opencv-python-headless

In [36]:
# 2: 확인
import ultralytics
ultralytics.checks()

Ultralytics 8.4.47 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 44.1/112.6 GB disk)


In [37]:
# 3: 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [38]:
# 4: BotSORT + 모자이크 + 로그
import cv2
from ultralytics import YOLO
import numpy as np
import csv
import torch



model = YOLO('/content/drive/MyDrive/DeepLearning/yolov8x-face-lindevs.onnx')

video_path = '/content/drive/MyDrive/DeepLearning/people.mp4'
cap = cv2.VideoCapture(video_path)
fps = cap.get(5)

out = cv2.VideoWriter('/content/drive/MyDrive/DeepLearning/output_botsort_face.mp4',
                      cv2.VideoWriter_fourcc(*'mp4v'), fps,
                      (int(cap.get(3)), int(cap.get(4))))

log_path = '/content/drive/MyDrive/DeepLearning/tracking_botsort_log.csv'
log_file = open(log_path, 'w', newline='')
writer = csv.writer(log_file)
writer.writerow(['frame', 'track_id', 'x1', 'y1', 'x2', 'y2', 'timestamp'])

frame_num = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame_num += 1
    timestamp = round(frame_num / fps, 3)

    # botsort로 트래킹
    results = model.track(frame, persist=True, tracker='botsort.yaml',
                          conf=0.5, verbose=False, device=0)

    if results[0].boxes.id is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy()
        ids = results[0].boxes.id.cpu().numpy()

        for box, track_id in zip(boxes, ids):
            x1, y1, x2, y2 = map(int, box)
            track_id = int(track_id)

            # 모자이크
            roi = frame[y1:y2, x1:x2]
            if roi.size > 0:
                blurred = cv2.GaussianBlur(roi, (51, 51), 0)
                frame[y1:y2, x1:x2] = blurred

            # ID 표시
            cv2.putText(frame, f'ID:{track_id}', (x1, y1-10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,0), 2)

            # 로그
            writer.writerow([frame_num, track_id, x1, y1, x2, y2, timestamp])

    out.write(frame)

cap.release()
out.release()
log_file.close()


True
Tesla T4
WARNING ⚠️ Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify','pose' or 'obb'.
Loading /content/drive/MyDrive/DeepLearning/yolov8x-face-lindevs.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.25.1 with CUDAExecutionProvider
1단계 완료!
